# Assignment 2 — Household Object Recognition, levelled up with SOTA text-to-video

**Topic (from A1):** classify household objects — `bottle`, `cup`, `key`, `pen` — from phone photos.

**What is new in A2**

| Step | What happens | Section |
|---|---|---|
| Dataset update | A1 photos, cleaned, de-duplicated and **balanced** (max 30 per class) | 2 |
| SOTA generator | **CogVideoX-2B** text→video (open-source; the A2 brief names Step-Video-T2V, which needs an 80 GB GPU — CogVideoX-2B is the open T2V model that fits a free Colab T4). Prompts ask for camera motion, blur and new viewpoints/backgrounds | 3 |
| Prompt → video → frames | Auto-sample frames from each clip, drop near-duplicates | 4 |
| Quality gate | **CLIP** zero-shot check: keep a synthetic frame only if CLIP agrees it shows the intended object | 5 |
| Experiments | 5-fold cross-validation on the **real** photos: A1 baseline vs + synthetic frames; ablations (CLIP filter, augmentation, backbone); clean **and** stress-test (motion blur, low light, rotation, …) metrics; FPS | 6–8 |
| Explainability | Grad-CAM + nearest-neighbour gallery | 9 |
| App bundle | Final model + gallery + labelled synthetic clips for the Gradio app | 10 |

Synthetic images are **never** used for testing — every score is measured on real photos only.

**Run:** Colab → *Runtime → Change runtime type → T4 GPU* → set `DATA_ROOT` below → *Run all*. Video generation is slow (minutes per clip); clips are saved to Drive one by one, so if the session drops just run all again — finished clips are skipped.

## 0. Setup

In [ ]:
import sys, subprocess, os
def _pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=False)
_pip("diffusers>=0.30", "transformers", "accelerate", "sentencepiece", "imageio", "imageio-ffmpeg", "scikit-image")
try:
    import pillow_heif; pillow_heif.register_heif_opener()
except ImportError:
    _pip("pillow-heif")
    try:
        import pillow_heif; pillow_heif.register_heif_opener()
    except Exception: pass
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
import json, time, random, copy, gc, shutil, zipfile, math, platform
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from PIL import Image, ImageOps, ImageDraw, ImageFont, ImageEnhance, ImageFilter, UnidentifiedImageError
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.transforms.functional as TF
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, confusion_matrix,
                             average_precision_score, classification_report)
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from skimage.feature import hog
from scipy import ndimage
import imageio.v2 as imageio
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})
sns.set_style("whitegrid")

### Configuration

In [ ]:
# ---------- paths (same Drive folder as A1) ----------
DATA_ROOT = Path(os.environ.get("CV_DATA_ROOT", "/content/drive/MyDrive/household_objects"))
RAW_DIR   = DATA_ROOT / "raw"
PROC_DIR  = DATA_ROOT / "processed"
A2_DIR    = Path(os.environ.get("CV_A2_DIR", DATA_ROOT / "a2"))
CLIP_DIR  = A2_DIR / "synthetic" / "clips"       # generated videos (one sub-folder per class)
FRAME_DIR = A2_DIR / "synthetic" / "frames"      # frames sampled from the videos
OUT_DIR   = A2_DIR / "results"                   # tables + figures for the report
BUNDLE_DIR= A2_DIR / "app_bundle"                # everything the Gradio app needs
for d in (CLIP_DIR, FRAME_DIR, OUT_DIR, BUNDLE_DIR): d.mkdir(parents=True, exist_ok=True)
assert RAW_DIR.exists(), f"Folder not found: {RAW_DIR}"

# ---------- data ----------
CLASSES = ["bottle", "cup", "key", "pen"]; NUM_CLASSES = len(CLASSES)
MAX_PER_CLASS = 30          # balanced real set (None = keep all)
IMG_SIZE, STORE_SIZE, SEED = 224, 320, 42

# ---------- text-to-video ----------
T2V_MODEL       = "THUDM/CogVideoX-2b"
CLIPS_PER_CLASS = 4         # 4 classes x 4 clips = 16 videos
T2V_STEPS       = 30        # denoising steps (50 = paper default; 30 is ~40% faster, slightly softer)
T2V_FRAMES      = 49        # CogVideoX-2B native length (6 s at 8 fps)
FRAMES_PER_CLIP = 8         # frames sampled from every clip for training
DUP_FRAME_BITS  = 1         # drop a sampled frame only if it is an exact repeat (dHash distance <= 1 of 64 bits)
CLIP_MODEL      = "openai/clip-vit-base-patch32"
CLIP_MIN_PROB   = 0.60      # keep a synthetic frame only if CLIP gives the intended class >= this

# ---------- training ----------
SMOKE_TEST = os.environ.get("CV_SMOKE_TEST", "0") == "1"   # quick code check only; keep False
PRETRAINED = not SMOKE_TEST
N_FOLDS    = 5
EPOCHS_HEAD, EPOCHS_FT = (1, 1) if SMOKE_TEST else (5, 10)
PATIENCE   = 5
BATCH_SIZE = 8 if SMOKE_TEST else 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
seed_everything()
ENV = {"device": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else platform.processor() or "CPU",
       "torch": torch.__version__, "python": platform.python_version(), "smoke_test": SMOKE_TEST}
print(ENV)

## 1. Plan of the SOTA pipeline

```
 text prompt ──► CogVideoX-2B (T5 text encoder → 3D-causal-VAE latent space → expert diffusion transformer)
                 │  49 frames, 720×480, camera motion / blur / new viewpoints
                 ▼
          video clip ──► sample 8 frames ──► drop near-duplicates (dHash)
                                              │
                                              ▼
                                   CLIP zero-shot quality gate (keep if CLIP agrees with the prompt's class)
                                              │
                 real photos (train folds) ───┴──► fine-tune ResNet-18 / EfficientNet-B0
                                                         │
                       real held-out fold ◄──────────────┘  clean + stress-test metrics
```
Why it helps this topic: in A1 bottles/cups were always shot standing in a kitchen and keys/pens lying flat on fabric or floors,
so a model could partly use the *scene*. The prompts below deliberately put **every object on every background** and add
the camera **motion, blur and viewpoints** that the real photos lack.

## 2. Real dataset (A1 photos) — clean, de-duplicate, balance

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".heic", ".heif"}
def dhash(img, size=8):
    a = np.asarray(img.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    return int("".join("1" if b else "0" for b in (a[:, 1:] > a[:, :-1]).flatten()), 2)
def hamming(a, b): return bin(int(a) ^ int(b)).count("1")
def resize_short(img, short=STORE_SIZE):
    w, h = img.size; s = short / min(w, h)
    return img if s >= 1 else img.resize((round(w * s), round(h * s)), Image.LANCZOS)

rec, bad = [], []
for cls in CLASSES:
    for p in sorted(q for q in (RAW_DIR / cls).rglob("*") if q.suffix.lower() in IMG_EXT):
        try: im = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
        except (UnidentifiedImageError, OSError) as e: bad.append(str(p)); continue
        out = PROC_DIR / cls / (p.stem + ".jpg")
        if not out.exists():
            out.parent.mkdir(parents=True, exist_ok=True); resize_short(im).save(out, quality=95)
        rec.append(dict(path=str(out), label=cls, width=im.size[0], height=im.size[1], hash=dhash(im)))
real = pd.DataFrame(rec)
n_raw = len(real)
drop = set()
for i in range(len(real)):
    if i in drop: continue
    for j in range(i + 1, len(real)):
        if j not in drop and hamming(real.hash[i], real.hash[j]) <= 4: drop.add(j)
real = real.drop(index=sorted(drop)).reset_index(drop=True)
n_dedup = len(real)
if MAX_PER_CLASS:
    keep = [i for _, g in real.groupby("label") for i in g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED).index]
    real = real.loc[sorted(keep)].reset_index(drop=True)
real["y"] = real.label.map({c: i for i, c in enumerate(CLASSES)})
real["synthetic"] = False
print(f"raw readable: {n_raw} | corrupted: {len(bad)} | after de-dup: {n_dedup} | after balancing: {len(real)}")
real_counts = real.label.value_counts().reindex(CLASSES)
real_counts.to_frame("real photos").T

In [ ]:
# Keep all real images in RAM (small, 320 px) — Drive is slow to read every epoch
def load(p): return Image.open(p).convert("RGB")
real["img"] = [load(p) for p in real.path]

## 3. SOTA generator: CogVideoX-2B text → video

Prompts are built from three lists — **object variants × backgrounds × camera motions** — so that each class appears on
every background (breaking the A1 scene shortcut) and with motion, blur and unusual viewpoints.

In [ ]:
OBJECTS = {
    "bottle": ["a clear plastic water bottle", "a red soda bottle", "a green glass bottle", "a metal sports water bottle"],
    "cup":    ["a white ceramic coffee mug", "a clear glass cup", "a paper coffee cup", "a blue mug with a handle"],
    "key":    ["a single silver door key with jagged teeth, macro close-up",
               "a single brass house key with a round head, macro close-up",
               "a single steel key with a blue rubber head, macro close-up",
               "a single long metal door key, macro close-up"],
    "pen":    ["a blue ballpoint pen", "a black gel pen", "a red marker pen", "a metal pen"],
}
BACKGROUNDS = ["on a wooden kitchen table", "on a grey carpet", "on a granite kitchen countertop",
               "on a white bed sheet", "held in a person's hand", "on a cluttered office desk"]
MOTIONS = ["A handheld phone camera slowly orbits around", "A shaky handheld camera quickly pans past",
           "The camera zooms in on", "A top-down camera slowly moves over", "A camera tilts down to"]
STYLE = "realistic home video, natural indoor light, slight motion blur, shallow depth of field, 4k"
NEG = "cartoon, illustration, text, letters, logo, watermark, padlock, lock, distorted, low quality, multiple objects"

def make_prompts():
    rng = random.Random(SEED); jobs = []
    for cls in CLASSES:
        for k in range(CLIPS_PER_CLASS):
            obj = OBJECTS[cls][k % len(OBJECTS[cls])]
            bg = BACKGROUNDS[(k + CLASSES.index(cls)) % len(BACKGROUNDS)]
            mo = MOTIONS[rng.randrange(len(MOTIONS))]
            jobs.append(dict(clip_id=f"{cls}_{k:02d}", label=cls, seed=SEED + 100 * CLASSES.index(cls) + k,
                             prompt=f"{mo} {obj} {bg}. {STYLE}."))
    return jobs
JOBS = make_prompts()
pd.DataFrame(JOBS)[["clip_id", "prompt"]].head(8)

In [ ]:
def clip_path(j): return CLIP_DIR / j["label"] / f"{j['clip_id']}.mp4"
def meta_path(j): return CLIP_DIR / j["label"] / f"{j['clip_id']}.json"
def done(j):   # a clip counts as done only if it was generated from the current prompt
    return clip_path(j).exists() and meta_path(j).exists() and json.load(open(meta_path(j)))["prompt"] == j["prompt"]
todo = [j for j in JOBS if not done(j)]
print(f"{len(JOBS) - len(todo)} clips already on Drive, {len(todo)} to generate")

In [ ]:
def encode_prompts(jobs):
    # Step A: run only the T5 text encoder, cache the embeddings, then free its ~9.5 GB
    from transformers import T5EncoderModel, T5Tokenizer
    from diffusers import CogVideoXPipeline
    tok = T5Tokenizer.from_pretrained(T2V_MODEL, subfolder="tokenizer")
    def run(dtype):
        te = T5EncoderModel.from_pretrained(T2V_MODEL, subfolder="text_encoder", torch_dtype=dtype,
                                            low_cpu_mem_usage=True).to(DEVICE)
        pipe = CogVideoXPipeline.from_pretrained(T2V_MODEL, text_encoder=te, tokenizer=tok,
                                                 transformer=None, vae=None, scheduler=None, torch_dtype=dtype)
        out = {}
        with torch.no_grad():
            for j in jobs:
                pe, ne = pipe.encode_prompt(j["prompt"], NEG, do_classifier_free_guidance=True,
                                            device=DEVICE, dtype=dtype)
                out[j["clip_id"]] = (pe.half().cpu(), ne.half().cpu())
        del pipe, te; gc.collect(); torch.cuda.empty_cache()
        return out
    emb = run(torch.float16)
    if any(torch.isnan(pe).any() or torch.isnan(ne).any() for pe, ne in emb.values()):
        print("fp16 T5 produced NaNs -> re-encoding in bfloat16"); emb = run(torch.bfloat16)
    return emb

def load_t2v():
    # Step B: diffusion transformer + VAE only (~4 GB in fp16)
    from diffusers import CogVideoXPipeline
    pipe = CogVideoXPipeline.from_pretrained(T2V_MODEL, text_encoder=None, tokenizer=None, torch_dtype=torch.float16)
    pipe.vae.enable_tiling(); pipe.vae.enable_slicing()
    # T4 has 15 GB: keep the transformer on the GPU only while it denoises, the VAE only while it decodes
    if torch.cuda.get_device_properties(0).total_memory < 20e9: pipe.enable_model_cpu_offload()
    else: pipe.to(DEVICE)
    return pipe

def fake_clip(j):
    # SMOKE_TEST only: imitates a clip (moving crop + blur of a real photo) so the rest of the code can be checked
    src = real[real.label == j["label"]].img.iloc[0].resize((480, 320)); frames = []
    for t in range(T2V_FRAMES):
        dx = int(60 * math.sin(t / 8)); fr = src.rotate(t * 2, translate=(dx, 0)).filter(ImageFilter.GaussianBlur(t % 3))
        frames.append(fr)
    return frames

def save_mp4(frames, path, fps=8):
    path.parent.mkdir(parents=True, exist_ok=True)
    imageio.mimsave(path, [np.asarray(f) for f in frames], fps=fps, quality=8, macro_block_size=16)

if todo:
    if SMOKE_TEST:
        for j in todo:
            save_mp4(fake_clip(j), clip_path(j)); json.dump({**j, "seconds": 0, "stub": True}, open(meta_path(j), "w"))
    else:
        assert DEVICE.type == "cuda", "Text-to-video needs a GPU runtime (Runtime -> Change runtime type -> T4 GPU)"
        emb = encode_prompts(todo)
        pipe = load_t2v(); t_start = time.time()
        for n, j in enumerate(todo, 1):
            t0 = time.time(); pe, ne = emb[j["clip_id"]]
            gen = lambda: pipe(prompt_embeds=pe.to(DEVICE), negative_prompt_embeds=ne.to(DEVICE),
                               num_inference_steps=T2V_STEPS, num_frames=T2V_FRAMES, guidance_scale=6.0,
                               generator=torch.Generator(device="cpu").manual_seed(j["seed"])).frames[0]
            try: frames = gen()
            except torch.cuda.OutOfMemoryError:     # last resort: stream weights layer by layer (slower)
                torch.cuda.empty_cache(); pipe.enable_sequential_cpu_offload(); frames = gen()
            save_mp4(frames, clip_path(j))
            sec = time.time() - t0
            json.dump({**j, "seconds": sec, "steps": T2V_STEPS, "model": T2V_MODEL}, open(meta_path(j), "w"))
            eta = (time.time() - t_start) / n * (len(todo) - n)
            print(f"[{n}/{len(todo)}] {j['clip_id']}  {sec/60:.1f} min  | ETA {eta/60:.0f} min")
        del pipe; gc.collect(); torch.cuda.empty_cache()

In [ ]:
gen_meta = pd.DataFrame([json.load(open(meta_path(j))) for j in JOBS if meta_path(j).exists()])
print(f"clips: {len(gen_meta)} | mean generation time: {gen_meta.seconds.mean()/60:.1f} min/clip")
# preview: first frame of one clip per class
fig, ax = plt.subplots(1, NUM_CLASSES, figsize=(4 * NUM_CLASSES, 3))
for a, cls in zip(ax, CLASSES):
    j = next(j for j in JOBS if j["label"] == cls)
    a.imshow(imageio.get_reader(clip_path(j)).get_data(T2V_FRAMES // 2)); a.axis("off")
    a.set_title(f"SYNTHETIC · {cls}", fontsize=10, color="crimson")
plt.suptitle("CogVideoX-2B clips (middle frame)"); plt.savefig(OUT_DIR / "fig_synthetic_clips.png"); plt.show()

## 4. Prompt → video → frames (auto-sampling)

In [ ]:
if FRAME_DIR.exists(): shutil.rmtree(FRAME_DIR)   # frames are re-sampled from the current clips every run
syn_rows = []
for j in JOBS:
    if not clip_path(j).exists(): continue
    vid = imageio.get_reader(clip_path(j)); n = vid.count_frames()
    idx = np.linspace(4, n - 5, FRAMES_PER_CLIP).round().astype(int)
    kept_hashes = []
    for k in idx:
        fr = Image.fromarray(vid.get_data(int(k))).convert("RGB")
        h = dhash(fr)
        if any(hamming(h, kh) <= DUP_FRAME_BITS for kh in kept_hashes): continue   # exact repeat
        kept_hashes.append(h)
        out = FRAME_DIR / j["label"] / f"{j['clip_id']}_f{k:02d}.jpg"
        out.parent.mkdir(parents=True, exist_ok=True)
        resize_short(fr).save(out, quality=95)
        syn_rows.append(dict(path=str(out), label=j["label"], clip_id=j["clip_id"], frame=int(k)))
syn = pd.DataFrame(syn_rows)
syn["y"] = syn.label.map({c: i for i, c in enumerate(CLASSES)}); syn["synthetic"] = True
syn["img"] = [load(p) for p in syn.path]
print(f"sampled synthetic frames: {len(syn)}"); syn.label.value_counts().reindex(CLASSES).to_frame("frames").T

## 5. CLIP quality gate (zero-shot)

Text-to-video models sometimes draw the wrong object, two objects, or a melted shape. CLIP (ViT-B/32) scores every frame
against `"a photo of a {bottle, cup, key, pen}"` **and** against "none of these" prompts (padlock, lock, metal box, text,
blurry dark photo, empty table); a frame is kept only if the intended class wins with probability ≥ `CLIP_MIN_PROB`.
(The first run showed why the distractors are needed: CogVideoX drew padlocks and a metal block with a fake logo for "key",
and with only the four class names CLIP still called them "key".)
The same CLIP model is also evaluated zero-shot on the real photos as an extra SOTA reference.

In [ ]:
CLIP_TEXTS = {"bottle": ["a photo of a bottle"], "cup": ["a photo of a cup", "a photo of a mug"],
              "key": ["a photo of a key"], "pen": ["a photo of a pen"]}
# "none of these" prompts: a frame that looks more like a padlock, a box, text or an empty blur is rejected
DISTRACTORS = ["a photo of a padlock", "a photo of a lock", "a photo of a metal box", "a photo of text",
               "a blurry dark photo", "an empty table"]
def _feat(o):   # transformers 4.x returns a tensor, 5.x an output object
    return o if torch.is_tensor(o) else o.pooler_output
class ClipScorer:
    def __init__(self):
        self.stub = SMOKE_TEST
        if self.stub: return
        from transformers import CLIPModel, CLIPProcessor
        self.model = CLIPModel.from_pretrained(CLIP_MODEL).to(DEVICE).eval()
        self.proc = CLIPProcessor.from_pretrained(CLIP_MODEL)
        texts = [t for c in CLASSES for t in CLIP_TEXTS[c]] + DISTRACTORS
        self.owner = np.array([CLASSES.index(c) for c in CLASSES for _ in CLIP_TEXTS[c]] + [NUM_CLASSES] * len(DISTRACTORS))
        with torch.no_grad():
            t = self.proc(text=texts, return_tensors="pt", padding=True).to(DEVICE)
            self.tf = F.normalize(_feat(self.model.get_text_features(**t)), dim=-1)
    @torch.no_grad()
    def probs(self, imgs, bs=32, with_other=False):
        # with_other=True -> NUM_CLASSES + 1 columns (last = "none of these"); False -> the 4 classes only
        k = NUM_CLASSES + 1 if with_other else NUM_CLASSES
        if self.stub:   # SMOKE_TEST: no internet model -> pretend CLIP is unsure
            return np.full((len(imgs), k), 1 / k)
        out = []
        for i in range(0, len(imgs), bs):
            x = self.proc(images=imgs[i:i + bs], return_tensors="pt").to(DEVICE)
            f = F.normalize(_feat(self.model.get_image_features(**x)), dim=-1)
            keep = np.ones(len(self.owner), bool) if with_other else self.owner < NUM_CLASSES
            logits = (100 * f @ self.tf[torch.as_tensor(keep, device=DEVICE)].T).softmax(-1).cpu().numpy()
            own = self.owner[keep]
            out.append(np.stack([logits[:, own == c].sum(1) for c in range(k)], 1))
        return np.concatenate(out)
clip_scorer = ClipScorer()

P = clip_scorer.probs(list(syn.img), with_other=True)
syn["clip_prob"] = P[np.arange(len(syn)), syn.y]
syn["clip_pred"] = P.argmax(1)
syn["keep"] = (syn.clip_pred == syn.y) & (syn.clip_prob >= CLIP_MIN_PROB)
if SMOKE_TEST: syn["keep"] = np.arange(len(syn)) % 4 != 0     # stub: drop every 4th frame
syn["clip_says"] = [(CLASSES + ["none of these"])[i] for i in syn.clip_pred]
gate = syn.groupby("label").agg(sampled=("keep", "size"), kept=("keep", "sum")).reindex(CLASSES)
gate["kept %"] = (100 * gate.kept / gate.sampled).round(1)
gate.to_csv(OUT_DIR / "clip_gate.csv"); gate

In [ ]:
fig, ax = plt.subplots(2, 8, figsize=(18, 5))
for r, flag in enumerate([True, False]):
    sub = syn[syn.keep == flag].sample(min(8, (syn.keep == flag).sum()), random_state=SEED)
    for a in ax[r]: a.axis("off")
    for a, (_, row) in zip(ax[r], sub.iterrows()):
        a.imshow(row.img); a.set_title(f"{row.label}: CLIP p={row.clip_prob:.2f}\n(CLIP: {row.clip_says})", fontsize=8,
                                      color="green" if flag else "crimson")
ax[0, 0].text(-0.1, 0.5, "KEPT", transform=ax[0, 0].transAxes, rotation=90, va="center", ha="right", fontsize=12)
ax[1, 0].text(-0.1, 0.5, "REJECTED", transform=ax[1, 0].transAxes, rotation=90, va="center", ha="right", fontsize=12)
plt.suptitle("SYNTHETIC frames after the CLIP quality gate"); plt.savefig(OUT_DIR / "fig_clip_gate.png"); plt.show()

## 6. Dataset update summary

In [ ]:
syn_kept = syn[syn.keep].reset_index(drop=True)
summary = pd.DataFrame({"real photos": real_counts,
                        "synthetic frames (sampled)": syn.label.value_counts().reindex(CLASSES).fillna(0).astype(int),
                        "synthetic frames (after CLIP)": syn_kept.label.value_counts().reindex(CLASSES).fillna(0).astype(int)})
summary.loc["total"] = summary.sum()
summary["synthetic share % (train pool)"] = (100 * summary["synthetic frames (after CLIP)"] /
                                             (summary["real photos"] + summary["synthetic frames (after CLIP)"])).round(1)
summary.to_csv(OUT_DIR / "dataset_update.csv"); summary

## 7. Stress test (robustness)

A1 already reached 100 % on clean test photos, so clean accuracy cannot show an improvement. The A2 brief asks for
*robustness*: every held-out real photo is also tested under 6 deterministic corruptions that imitate real phone problems.

In [ ]:
def motion_blur(img, length=21, angle=0.0):
    k = np.zeros((length, length), np.float32); k[length // 2, :] = 1
    k = np.asarray(Image.fromarray((k * 255).astype(np.uint8)).rotate(angle, resample=Image.BILINEAR), np.float32)
    k /= k.sum(); a = np.asarray(img, np.float32)
    out = np.stack([ndimage.convolve(a[..., c], k, mode="reflect") for c in range(3)], -1)
    return Image.fromarray(out.clip(0, 255).astype(np.uint8))
def low_light(img, seed=0):
    a = np.asarray(ImageEnhance.Brightness(img).enhance(0.3), np.float32)
    a += np.random.default_rng(seed).normal(0, 8, a.shape)
    return Image.fromarray(a.clip(0, 255).astype(np.uint8))
def jpeg(img, q=8):
    import io; b = io.BytesIO(); img.save(b, "JPEG", quality=q); b.seek(0); return Image.open(b).convert("RGB")
def occlude(img, seed=0):
    rng = np.random.default_rng(seed); w, h = img.size; img = img.copy(); d = ImageDraw.Draw(img)
    bw, bh = int(w * 0.45), int(h * 0.45); x, y = rng.integers(0, w - bw), rng.integers(0, h - bh)
    d.rectangle([x, y, x + bw, y + bh], fill=(128, 128, 128)); return img
def perspective(img, s=0.35):
    w, h = img.size; dx, dy = int(w * s / 2), int(h * s / 2)
    return TF.perspective(img, [[0, 0], [w, 0], [w, h], [0, h]], [[dx, dy], [w - dx, 0], [w, h], [0, h - dy]], fill=0)

CORRUPTIONS = {
    "motion blur": lambda im, i: motion_blur(im, 21, angle=(i * 37) % 180),
    "defocus blur": lambda im, i: im.filter(ImageFilter.GaussianBlur(4)),
    "low light + noise": lambda im, i: low_light(im, i),
    "rotation 60°": lambda im, i: im.rotate(60, expand=True),
    "viewpoint (perspective)": lambda im, i: perspective(im),
    "occlusion 45%": lambda im, i: occlude(im, i),
    "JPEG q=8": lambda im, i: jpeg(im),
}
ex = real.img.iloc[0]
fig, ax = plt.subplots(1, len(CORRUPTIONS) + 1, figsize=(2.4 * (len(CORRUPTIONS) + 1), 2.6))
ax[0].imshow(ex); ax[0].set_title("clean", fontsize=9)
for a, (n, f) in zip(ax[1:], CORRUPTIONS.items()): a.imshow(f(ex, 0)); a.set_title(n, fontsize=9)
for a in ax: a.axis("off")
plt.savefig(OUT_DIR / "fig_corruptions.png"); plt.show()
# corrupted copies are deterministic -> compute once, reuse for every model
STRESSED = {c: [f(im, i) for i, im in enumerate(real.img)] for c, f in CORRUPTIONS.items()}

## 8. Experiments — 5-fold cross-validation on real photos

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20), transforms.ColorJitter(0.3, 0.3, 0.3, 0.05), transforms.RandomGrayscale(0.1),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD), transforms.RandomErasing(p=0.25, scale=(0.02, 0.15))])
eval_tf = transforms.Compose([transforms.Resize(IMG_SIZE), transforms.CenterCrop(IMG_SIZE),
                              transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class ImgDS(Dataset):
    def __init__(self, imgs, ys, tf): self.imgs, self.ys, self.tf = list(imgs), list(ys), tf
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.tf(self.imgs[i]), self.ys[i]
def loader(imgs, ys, tf, shuffle):
    g = torch.Generator(); g.manual_seed(SEED)
    return DataLoader(ImgDS(imgs, ys, tf), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0, generator=g)

def build(arch):
    if arch == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, NUM_CLASSES)); head = m.fc
    else:
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if PRETRAINED else None)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, NUM_CLASSES)); head = m.classifier
    return m.to(DEVICE), head

@torch.no_grad()
def predict_probs(model, imgs):
    model.eval(); out = []
    for x, _ in loader(imgs, [0] * len(imgs), eval_tf, False): out.append(model(x.to(DEVICE)).softmax(1).cpu())
    return torch.cat(out).numpy()

def fit(arch, tr_imgs, tr_ys, va_imgs, va_ys, aug=True):
    seed_everything(); model, head = build(arch); hid = {id(p) for p in head.parameters()}
    crit = nn.CrossEntropyLoss(label_smoothing=0.1); hist = []
    tl = loader(tr_imgs, tr_ys, train_tf if aug else eval_tf, True)
    best, best_state, bad = -1, None, 0
    for stage, epochs in (("head", EPOCHS_HEAD), ("finetune", EPOCHS_FT)):
        for p in model.parameters(): p.requires_grad = stage == "finetune" or id(p) in hid
        if stage == "head": opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4); sch = None
        else:
            opt = torch.optim.AdamW([{"params": [p for p in model.parameters() if id(p) not in hid], "lr": 1e-4},
                                     {"params": head.parameters(), "lr": 1e-3}], weight_decay=1e-4)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, epochs))
        for ep in range(epochs):
            model.train(); s, c, n = 0.0, 0, 0
            for x, y in tl:
                x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad(); o = model(x); l = crit(o, y); l.backward(); opt.step()
                s += l.item() * len(y); c += (o.argmax(1) == y).sum().item(); n += len(y)
            if sch: sch.step()
            pv = predict_probs(model, va_imgs); vy = np.array(va_ys)
            vl = F.nll_loss(torch.log(torch.tensor(pv) + 1e-8), torch.tensor(vy)).item()
            vf1 = precision_recall_fscore_support(vy, pv.argmax(1), average="macro", zero_division=0)[2]
            hist.append(dict(stage=stage, train_loss=s / n, train_acc=c / n, val_loss=vl, val_acc=(pv.argmax(1) == vy).mean(), val_f1=vf1))
            if vf1 > best: best, best_state, bad = vf1, copy.deepcopy(model.state_dict()), 0
            else:
                bad += 1
                if stage == "finetune" and bad >= PATIENCE: break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist)

In [ ]:
def metrics(y, P):
    yp = P.argmax(1); p, r, f1, _ = precision_recall_fscore_support(y, yp, average="macro", zero_division=0)
    Y = label_binarize(y, classes=range(NUM_CLASSES))
    return dict(accuracy=accuracy_score(y, yp), precision=p, recall=r, f1=f1,
                mAP=average_precision_score(Y, P, average="macro"))

CONDITIONS = {   # name: (backbone, synthetic frames used for training, augmentation)
    "A1 baseline: ResNet-18, real only":        ("resnet18", None, True),
    "SOTA: ResNet-18 + synthetic (CLIP-gated)": ("resnet18", "gated", True),
    "Ablation: + synthetic, no CLIP gate":      ("resnet18", "all", True),
    "Ablation: real only, no augmentation":     ("resnet18", None, False),
    "Control: EfficientNet-B0, real only":       ("efficientnet_b0", None, True),
    "SOTA: EfficientNet-B0 + synthetic (gated)":("efficientnet_b0", "gated", True),
}
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(real, real.y))
pd.DataFrame({"fold": range(N_FOLDS), "test images": [len(te) for _, te in folds]}).T

In [ ]:
RES, HIST = {}, {}
t_all = time.time()
for name, (arch, syn_mode, aug) in CONDITIONS.items():
    P_clean = np.zeros((len(real), NUM_CLASSES)); P_stress = {c: np.zeros((len(real), NUM_CLASSES)) for c in CORRUPTIONS}
    fold_scores = []; t0 = time.time()
    for k, (tr, te) in enumerate(folds):
        tr_df = real.iloc[tr]
        tr_df, va_df = train_test_split(tr_df, test_size=0.15, stratify=tr_df.y, random_state=SEED)
        imgs, ys = list(tr_df.img), list(tr_df.y)
        if syn_mode:
            s = syn_kept if syn_mode == "gated" else syn
            imgs += list(s.img); ys += list(s.y)
        model, hist = fit(arch, imgs, ys, list(va_df.img), list(va_df.y), aug)
        if k == 0: HIST[name] = hist
        te_df = real.iloc[te]
        P_clean[te] = predict_probs(model, list(te_df.img))
        for c, f in CORRUPTIONS.items():
            P_stress[c][te] = predict_probs(model, [STRESSED[c][i] for i in te])
        fold_scores.append(metrics(te_df.y.values, P_clean[te])["f1"])
        del model; torch.cuda.empty_cache()
    stress = {c: metrics(real.y.values, P)["accuracy"] for c, P in P_stress.items()}
    stress_all = metrics(np.tile(real.y.values, len(CORRUPTIONS)), np.concatenate(list(P_stress.values())))
    RES[name] = dict(clean=metrics(real.y.values, P_clean), f1_fold_std=np.std(fold_scores),
                     stress=stress, stress_all=stress_all, P_clean=P_clean, P_stress=P_stress,
                     train_s=(time.time() - t0) / N_FOLDS)
    print(f"{name:45s} clean F1 {RES[name]['clean']['f1']:.3f} | stress acc {stress_all['accuracy']:.3f} | {time.time()-t0:.0f}s")
print(f"all conditions: {(time.time()-t_all)/60:.1f} min")

### 8.1 Reference baselines: HOG + SVM (A1 classical) and CLIP zero-shot (no training)

In [ ]:
def hog_feat(im):
    im = ImageOps.fit(im, (128, 128)); g = np.asarray(im.convert("L"), np.float32) / 255
    h = hog(g, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), block_norm="L2-Hys")
    hsv = np.asarray(im.convert("HSV"))
    return np.concatenate([h, 10 * np.concatenate([np.histogram(hsv[..., c], 16, (0, 255), density=True)[0] for c in range(3)])])
X = np.stack([hog_feat(im) for im in real.img]); y = real.y.values
P_clean = np.zeros((len(real), NUM_CLASSES)); P_stress = {c: np.zeros_like(P_clean) for c in CORRUPTIONS}
for tr, te in folds:
    gs = GridSearchCV(Pipeline([("s", StandardScaler()), ("p", PCA(0.95, random_state=SEED)),
                                ("c", SVC(probability=True, random_state=SEED))]),
                      {"c__C": [1, 10, 100]}, cv=3, scoring="f1_macro").fit(X[tr], y[tr])
    P_clean[te] = gs.predict_proba(X[te])
    for c, f in CORRUPTIONS.items():
        P_stress[c][te] = gs.predict_proba(np.stack([hog_feat(STRESSED[c][i]) for i in te]))
RES["A1 classical: HOG + SVM"] = dict(clean=metrics(y, P_clean), f1_fold_std=np.nan,
    stress={c: metrics(y, P)["accuracy"] for c, P in P_stress.items()},
    stress_all=metrics(np.tile(y, len(CORRUPTIONS)), np.concatenate(list(P_stress.values()))), train_s=np.nan)

if not SMOKE_TEST:
    Pc = clip_scorer.probs(list(real.img))
    Ps = {c: clip_scorer.probs(STRESSED[c]) for c in CORRUPTIONS}
    RES["Zero-shot CLIP ViT-B/32 (no training)"] = dict(clean=metrics(y, Pc), f1_fold_std=np.nan,
        stress={c: metrics(y, P)["accuracy"] for c, P in Ps.items()},
        stress_all=metrics(np.tile(y, len(CORRUPTIONS)), np.concatenate(list(Ps.values()))), train_s=0)

### 8.2 Results tables

In [ ]:
rows = []
for n, r in RES.items():
    rows.append({"Model": n, "Clean Acc": r["clean"]["accuracy"], "Clean macro-F1": r["clean"]["f1"],
                 "F1 fold SD": r["f1_fold_std"], "Clean mAP": r["clean"]["mAP"],
                 "Stress Acc": r["stress_all"]["accuracy"], "Stress macro-F1": r["stress_all"]["f1"],
                 "Stress mAP": r["stress_all"]["mAP"], "Train time / fold (s)": r["train_s"]})
main = pd.DataFrame(rows).set_index("Model")
main.to_csv(OUT_DIR / "results_main.csv")
display(main.style.format("{:.3f}", na_rep="—").format({"Train time / fold (s)": "{:.0f}"}, na_rep="—")
        .highlight_max(subset=["Stress Acc", "Stress macro-F1"], color="#c6efce"))
stress_tab = pd.DataFrame({n: r["stress"] for n, r in RES.items()}).T
stress_tab.to_csv(OUT_DIR / "results_stress_by_corruption.csv")
display(stress_tab.style.format("{:.2f}").background_gradient(cmap="RdYlGn", vmin=0.25, vmax=1, axis=None))
try: print(main.round(3).to_markdown())
except ImportError: pass

In [ ]:
base, sota = "A1 baseline: ResNet-18, real only", "SOTA: ResNet-18 + synthetic (CLIP-gated)"
delta = (stress_tab.loc[sota] - stress_tab.loc[base]).rename("Δ accuracy (SOTA − baseline)")
ax = stress_tab.loc[[base, sota]].T.plot.barh(figsize=(8, 4.5), width=0.75, color=["#9aa5b1", "#2b6cb0"])
ax.set_xlim(0, 1.05); ax.set_xlabel("accuracy on held-out real photos"); ax.legend(loc="lower right", fontsize=8)
ax.set_title("Robustness: baseline vs + synthetic video frames")
plt.savefig(OUT_DIR / "fig_stress_baseline_vs_sota.png"); plt.show()
delta.to_frame()

In [ ]:
names = [base, sota]
fig, ax = plt.subplots(2, 2, figsize=(10, 7))
for c, n in enumerate(names):
    h = HIST[n]; ep = np.arange(1, len(h) + 1); s = ep[(h.stage == "finetune").values][0] - 0.5
    ax[0, c].plot(ep, h.train_loss, label="train"); ax[0, c].plot(ep, h.val_loss, label="val")
    ax[1, c].plot(ep, h.train_acc, label="train"); ax[1, c].plot(ep, h.val_acc, label="val")
    for a in ax[:, c]: a.axvline(s, ls="--", c="gray", lw=1); a.legend()
    ax[0, c].set_title(f"{n}\nloss (fold 1)", fontsize=9); ax[1, c].set_title("accuracy", fontsize=9); ax[1, c].set_xlabel("epoch")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_learning_curves.png"); plt.show()

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(9, 8))
for r, n in enumerate(names):
    for c, (title, P) in enumerate([("clean", RES[n]["P_clean"]), ("motion blur", RES[n]["P_stress"]["motion blur"])]):
        cm = confusion_matrix(real.y, P.argmax(1), labels=range(NUM_CLASSES))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, xticklabels=CLASSES, yticklabels=CLASSES, ax=ax[r, c])
        ax[r, c].set_title(f"{n.split(':')[0]} — {title}", fontsize=9); ax[r, c].set_xlabel("predicted"); ax[r, c].set_ylabel("true")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_confusion.png"); plt.show()

### 8.3 Speed (FPS)

In [ ]:
def latency_ms(model, device, n=50):
    model = model.to(device).eval(); x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    with torch.no_grad():
        for _ in range(5): model(x)
        if device.type == "cuda": torch.cuda.synchronize()
        t = time.time()
        for _ in range(n): model(x)
        if device.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t) / n * 1000
speed = []
for arch in ["resnet18", "efficientnet_b0"]:
    m, _ = build(arch)
    for dev in ([DEVICE] if DEVICE.type == "cpu" else [DEVICE, torch.device("cpu")]):
        ms = latency_ms(m, dev, 10 if SMOKE_TEST else 50)
        speed.append({"backbone": arch, "device": ENV["device"] if dev.type == "cuda" else "CPU (Colab)",
                      "ms / image": round(ms, 2), "FPS": round(1000 / ms, 1)})
speed = pd.DataFrame(speed); speed.to_csv(OUT_DIR / "results_speed.csv"); speed

## 9. Final model, Grad-CAM and nearest neighbours

In [ ]:
FINAL = "SOTA: ResNet-18 + synthetic (CLIP-gated)"
arch = CONDITIONS[FINAL][0]
tr_df, va_df = train_test_split(real, test_size=0.15, stratify=real.y, random_state=SEED)
final_model, final_hist = fit(arch, list(tr_df.img) + list(syn_kept.img), list(tr_df.y) + list(syn_kept.y),
                              list(va_df.img), list(va_df.y))
final_model.eval(); print("final model trained on", len(tr_df), "real +", len(syn_kept), "synthetic images")

class GradCAM:
    def __init__(self, model, layer):
        self.m = model; self.a = self.g = None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "a", o.detach()))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "g", go[0].detach()))
    def __call__(self, x):
        self.m.eval(); out = self.m(x); c = out.argmax(1).item(); self.m.zero_grad(); out[0, c].backward()
        cam = F.relu((self.g.mean((2, 3), keepdim=True) * self.a).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        return ((cam - cam.min()) / (cam.max() - cam.min() + 1e-8)).cpu().numpy(), c
layer = final_model.layer4[-1] if arch == "resnet18" else final_model.features[-1]
cam = GradCAM(final_model, layer)
def denorm(t): return (t.permute(1, 2, 0).cpu().numpy() * STD + MEAN).clip(0, 1)
fig, ax = plt.subplots(2, 2 * NUM_CLASSES, figsize=(2.3 * 2 * NUM_CLASSES, 5))
for c, cls in enumerate(CLASSES):
    im = va_df[va_df.label == cls].img.iloc[0] if (va_df.label == cls).any() else real[real.label == cls].img.iloc[0]
    for r, (tag, img) in enumerate([("clean", im), ("motion blur", CORRUPTIONS["motion blur"](im, 3))]):
        x = eval_tf(img).unsqueeze(0).to(DEVICE); h, pc = cam(x)
        ax[r, 2 * c].imshow(denorm(x[0].detach())); ax[r, 2 * c].set_title(f"{cls} · {tag}", fontsize=8)
        ax[r, 2 * c + 1].imshow(denorm(x[0].detach())); ax[r, 2 * c + 1].imshow(h, cmap="jet", alpha=0.45)
        ax[r, 2 * c + 1].set_title(f"pred: {CLASSES[pc]}", fontsize=8)
for a in ax.flat: a.axis("off")
plt.suptitle("Grad-CAM of the final model (clean vs motion-blurred)"); plt.savefig(OUT_DIR / "fig_gradcam.png"); plt.show()

In [ ]:
@torch.no_grad()
def embed(model, imgs):
    body = nn.Sequential(*list(model.children())[:-1]) if arch == "resnet18" else nn.Sequential(model.features, model.avgpool)
    out = []
    for x, _ in loader(imgs, [0] * len(imgs), eval_tf, False):
        out.append(F.normalize(body(x.to(DEVICE)).flatten(1), dim=1).cpu())
    return torch.cat(out).numpy()
gal = pd.concat([real[["img", "label", "synthetic"]], syn_kept[["img", "label", "synthetic"]]], ignore_index=True)
E = embed(final_model, list(gal.img))
q = va_df.img.iloc[0]; qe = embed(final_model, [q])[0]; nn_idx = np.argsort(-E @ qe)[:6]
fig, ax = plt.subplots(1, 7, figsize=(17, 2.8)); ax[0].imshow(q); ax[0].set_title("query", fontsize=9)
for a, i in zip(ax[1:], nn_idx):
    a.imshow(gal.img[i]); a.set_title(f"{gal.label[i]}{' · SYNTHETIC' if gal.synthetic[i] else ''}\nsim {E[i] @ qe:.2f}",
                                      fontsize=8, color="crimson" if gal.synthetic[i] else "black")
for a in ax: a.axis("off")
plt.savefig(OUT_DIR / "fig_nearest_neighbours.png"); plt.show()

## 10. App bundle (for the Gradio app) and export

In [ ]:
def watermark(fr, text="SYNTHETIC · generated with CogVideoX-2B"):
    fr = fr.copy(); d = ImageDraw.Draw(fr); w, h = fr.size
    try: font = ImageFont.load_default(size=max(14, h // 22))
    except TypeError: font = ImageFont.load_default()
    d.rectangle([0, h - h // 11, w, h], fill=(180, 0, 0)); d.text((10, h - h // 11 + 4), text, fill="white", font=font)
    return fr

if BUNDLE_DIR.exists(): shutil.rmtree(BUNDLE_DIR)
(BUNDLE_DIR / "gallery").mkdir(parents=True); (BUNDLE_DIR / "synthetic").mkdir(); (BUNDLE_DIR / "examples").mkdir()
for cls in CLASSES:   # one held-out real photo per class as clickable examples in the app
    src = va_df[va_df.label == cls] if (va_df.label == cls).any() else real[real.label == cls]
    src.img.iloc[0].save(BUNDLE_DIR / "examples" / f"{cls}.jpg", quality=90)
torch.save(final_model.state_dict(), BUNDLE_DIR / "model.pt")
gmeta = []
for i, row in gal.iterrows():
    fn = f"gallery/{i:04d}.jpg"; ImageOps.fit(row.img, (160, 160)).save(BUNDLE_DIR / fn, quality=85)
    gmeta.append({"file": fn, "label": row.label, "synthetic": bool(row.synthetic)})
np.save(BUNDLE_DIR / "embeddings.npy", E.astype(np.float32))
json.dump(gmeta, open(BUNDLE_DIR / "gallery.json", "w"))
smeta = []
for j in JOBS:   # one watermarked clip per (class, first 2 clips)
    if not clip_path(j).exists() or int(j["clip_id"][-2:]) >= 2: continue
    frames = [watermark(Image.fromarray(f)) for f in imageio.mimread(clip_path(j), memtest=False)]
    out = BUNDLE_DIR / "synthetic" / f"{j['clip_id']}.mp4"; save_mp4(frames, out)
    smeta.append({"file": f"synthetic/{out.name}", "label": j["label"], "prompt": j["prompt"], "model": T2V_MODEL})
json.dump(smeta, open(BUNDLE_DIR / "synthetic.json", "w"), indent=1)
meta = {"classes": CLASSES, "arch": arch, "img_size": IMG_SIZE, "mean": MEAN, "std": STD,
        "trained_on": {"real": int(len(tr_df)), "synthetic": int(len(syn_kept))}, "condition": FINAL,
        "cv_metrics": {k: {m: float(v) for m, v in RES[FINAL][k].items()} for k in ("clean", "stress_all")},
        "t2v_model": T2V_MODEL, "clip_model": CLIP_MODEL, "train_env": ENV}
json.dump(meta, open(BUNDLE_DIR / "meta.json", "w"), indent=1)
zf = A2_DIR / "app_bundle.zip"
with zipfile.ZipFile(zf, "w", zipfile.ZIP_DEFLATED) as z:
    for p in BUNDLE_DIR.rglob("*"): z.write(p, p.relative_to(BUNDLE_DIR.parent))
print(f"bundle: {zf} ({zf.stat().st_size/1e6:.1f} MB)")

In [ ]:
export = {"env": ENV, "dataset": summary.reset_index().to_dict("records"),
          "clip_gate": gate.reset_index().to_dict("records"),
          "generation": {"clips": int(len(gen_meta)), "min_per_clip": float(gen_meta.seconds.mean() / 60),
                         "steps": T2V_STEPS, "frames": T2V_FRAMES, "model": T2V_MODEL},
          "results": {n: {"clean": r["clean"], "stress_all": r["stress_all"], "stress": r["stress"],
                          "f1_fold_std": None if pd.isna(r["f1_fold_std"]) else float(r["f1_fold_std"])} for n, r in RES.items()},
          "speed": speed.to_dict("records"), "prompts": JOBS}
json.dump(export, open(OUT_DIR / "a2_metrics.json", "w"), indent=1, default=float)
print("saved:", sorted(p.name for p in OUT_DIR.iterdir()))

## 11. Try the app right here (optional)
Download `app.py` and `requirements.txt` from the project repo into `/content`, then run the cell below.
Gradio prints a public `*.gradio.live` link (valid ~72 h) that works on any phone or laptop.

In [ ]:
# !pip -q install gradio
# !python /content/app.py --bundle "{BUNDLE_DIR}" --share